# Бросок камня: откуда берётся дифференциальное уравнение

**Задача.** Перебросить камень через стену: пройти в бойницу (высота **8–11 м** на расстоянии **15 м**) и попасть в цель на земле в **25 м**. Управлять можно только начальной скоростью и углом. Два условия, два неизвестных — задача поставлена корректно. Вопрос в том, чем её решать.

**Как запускать.** «Среда выполнения» → «Выполнить всё» (Ctrl+F9), затем крутить ползунки под графиком. Устанавливать ничего не нужно.

---

### Сценарий: нажимать по порядку

1. Поставьте **«Трение: нет»** и нажмите **«Школьный прицел»**. Это точный ответ задачи, полученный из формулы параболы: $v_0 = 16{,}48$ м/с, угол $57{,}72^\circ$. Камень проходит по центру бойницы и ложится ровно в цель.

2. **Не трогая прицел**, переключите трение на **квадратичное**. Камень приходит к стене на высоте **4,25 м** вместо 8–11 и разбивается. Точный ответ точной формулы промахнулся по вертикали на 3,7 м.

3. Попробуйте попасть руками. Правая панель показывает промах по каждому из двух условий: нужен ноль у **обеих** кривых одновременно — а одним углом этого не добиться.

4. Когда надоест — **«Пристреляться»**. Это метод стрельбы: школьный прицел берётся начальным приближением и уточняется Ньютоном. Ответ: $v_0 = 20{,}22$ м/с, угол $51{,}72^\circ$.

---

### Зачем это в начале курса

Трение здесь — не поправка к ответу, а причина, по которой ответа в виде формулы больше нет:

| закон сопротивления | уравнение | есть замкнутое решение? |
|---|---|---|
| нет | $\dot{\vec v} = \vec g$ | **да** — парабола, школьная формула |
| линейное | $\dot{\vec v} = \vec g - \beta(\vec v - \vec w)$ | **да** — система распадается на два *независимых* скалярных уравнения |
| квадратичное | $\dot{\vec v} = \vec g - \frac{k}{m}\lvert\vec v - \vec w\rvert(\vec v - \vec w)$ | **нет** — множитель $\lvert\vec v\rvert$ смешивает компоненты |

Средняя строка — ровно то, чем занимается это занятие: два уравнения первого порядка, каждое решается разделением переменных. Нижняя строка — то, зачем нужен весь остальной курс.

Проверить среднюю строку можно глазами: при линейном трении на численную кривую ложатся оранжевые кружки — это подстановка $t$ в замкнутую формулу
$$x(t) = w_0 t + \frac{v_{x0}-w_0}{\beta}\left(1-e^{-\beta t}\right), \qquad y(t) = \frac{v_{y0} + g/\beta}{\beta}\left(1-e^{-\beta t}\right) - \frac{g}{\beta}\,t .$$
Совпали — значит, и формула верна, и решатель не врёт.

In [ ]:
# =============================================================
#  БРОСОК КАМНЯ — движок: физика, решатель, пристрелка.
#  Интерфейс — в следующей ячейке. Здесь ничего не рисуется.
# =============================================================
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import root

G = 9.81                      # м/с^2

# ---------------- сцена (метры) ----------------
WALL_X = 15.0                 # плоскость стены
SLIT_LO, SLIT_HI = 8.0, 11.0  # бойница
TARGET_X, TARGET_R = 25.0, 1.0
SLIT_MID = 0.5 * (SLIT_LO + SLIT_HI)

# ---------------- законы сопротивления ----------------
LAW_NONE, LAW_LIN, LAW_QUAD = 0, 1, 2


def drag_coeff(law, v_T):
    """Коэффициент трения из терминальной скорости v_T.

    Так два закона сравниваются честно: при одинаковой v_T камень
    в свободном падении выходит на одну и ту же скорость, а разница
    в траектории — целиком заслуга ЗАКОНА, а не силы трения.
        линейное:      dv/dt = -g - beta*v      -> v_T = g/beta
        квадратичное:  dv/dt = -g - (k/m)*|v|v  -> v_T = sqrt(g/(k/m))
    """
    if law == LAW_LIN:
        return G / v_T                 # beta,  1/с
    if law == LAW_QUAD:
        return G / v_T ** 2            # k/m,   1/м
    return 0.0


def air_speed(y, w0, shear):
    """Горизонтальная скорость ВОЗДУХА на высоте y."""
    return w0 + shear * y


def rhs(t, s, law, c, w0, shear):
    """Правая часть системы. Трение зависит от скорости камня
    ОТНОСИТЕЛЬНО воздуха — только так ветер входит в уравнение."""
    x, y, vx, vy = s
    ux = vx - air_speed(y, w0, shear)      # относительная скорость
    uy = vy
    if law == LAW_NONE:
        return [vx, vy, 0.0, -G]
    if law == LAW_LIN:
        return [vx, vy, -c * ux, -G - c * uy]
    u = np.hypot(ux, uy)
    return [vx, vy, -c * u * ux, -G - c * u * uy]


# ---------------- события: земля и плоскость стены ----------------
def _ev_ground(t, s, *a):
    return s[1]


_ev_ground.terminal = True
_ev_ground.direction = -1          # только сверху вниз


def _ev_wall(t, s, *a):
    return s[0] - WALL_X           # не terminal: сначала высота, потом вердикт


def integrate(v0, ang_deg, law, v_T, w0=0.0, shear=0.0, rtol=1e-8, t_max=25.0):
    a = np.radians(ang_deg)
    s0 = [0.0, 0.0, v0 * np.cos(a), v0 * np.sin(a)]
    return solve_ivp(rhs, [0, t_max], s0,
                     args=(law, drag_coeff(law, v_T), w0, shear),
                     events=[_ev_ground, _ev_wall],
                     dense_output=True, rtol=rtol, atol=1e-10)


def analyse(sol, n_pts=600):
    """Разбор решения: высота на стене, удар, приземление, апогей."""
    r = {'y_wall': None, 't_crash': None, 'y_crash': None, 'x_land': None}
    for te, ye in zip(sol.t_events[1], sol.y_events[1]):
        if r['y_wall'] is None:
            r['y_wall'] = ye[1]                     # первое пересечение стены
        if not (SLIT_LO <= ye[1] <= SLIT_HI):
            r['t_crash'], r['y_crash'] = te, ye[1]  # мимо бойницы -> удар
            break
    if sol.status == 1 and len(sol.y_events[0]):
        r['x_land'] = sol.y_events[0][0][0]
    r['crashed'] = r['t_crash'] is not None
    t_end = r['t_crash'] if r['crashed'] else sol.t[-1]
    tt = np.linspace(0.0, t_end, n_pts)
    xy = sol.sol(tt)
    r['t'], r['x'], r['y'] = tt, xy[0], xy[1]
    r['apex'] = xy[1].max()                          # апогей ПРОЙДЕННОЙ части
    r['v_end'] = float(np.hypot(xy[2][-1], xy[3][-1]))
    r['t_end'] = t_end
    r['hit'] = (not r['crashed'] and r['x_land'] is not None
                and abs(r['x_land'] - TARGET_X) <= TARGET_R)
    return r


# ---------------- точные решения ----------------
def parabola(v0, ang_deg, n=400):
    """Школьная парабола (k=0) для тех же начальных условий."""
    a = np.radians(ang_deg)
    vx0, vy0 = v0 * np.cos(a), v0 * np.sin(a)
    t = np.linspace(0.0, 2.0 * vy0 / G, n)
    return vx0 * t, vy0 * t - 0.5 * G * t ** 2


def linear_exact(v0, ang_deg, v_T, w0, t_end, n=400):
    """ЛИНЕЙНОЕ трение решается в замкнутом виде: система распадается
    на два НЕЗАВИСИМЫХ скалярных уравнения (это материал П1/П4)
        vx' = -beta*(vx - w0),      vy' = -g - beta*vy
    Квадратичное так не распадается: |v| смешивает компоненты."""
    b = G / v_T
    a = np.radians(ang_deg)
    vx0, vy0 = v0 * np.cos(a), v0 * np.sin(a)
    t = np.linspace(0.0, t_end, n)
    E = np.exp(-b * t)
    x = w0 * t + (vx0 - w0) / b * (1.0 - E)
    y = (vy0 + G / b) / b * (1.0 - E) - G / b * t
    return x, y


def school_aim():
    """Прицел по школьной формуле: два условия на два неизвестных,
    решается ТОЧНО, потому что при k=0 траектория — парабола.
        y(TARGET_X) = 0            (попасть в цель)
        y(WALL_X)   = SLIT_MID     (пройти по центру бойницы)
    """
    u = SLIT_MID / (WALL_X * (1.0 - WALL_X / TARGET_X))       # u = tg(theta)
    v0 = np.sqrt(G * (1.0 + u ** 2) * TARGET_X / (2.0 * u))
    return v0, np.degrees(np.arctan(u))


# ---------------- метод стрельбы ----------------
def residual(p, law, v_T, w0, shear, rtol=1e-7):
    """Невязка краевой задачи: F1 — промах по бойнице, F2 — по цели.
    При пристрелке стена — УСЛОВИЕ, а не препятствие (иначе невязка рвётся)."""
    v0, ang = p
    if v0 <= 0.5:
        return [1e4 * (1.0 - v0)] * 2
    sol = integrate(v0, ang, law, v_T, w0, shear, rtol=rtol)
    x_land = (sol.y_events[0][0][0] if (sol.status == 1 and len(sol.y_events[0]))
              else sol.y[0][-1])
    if len(sol.y_events[1]):
        F1 = sol.y_events[1][0][1] - SLIT_MID
    else:                       # не достал до стены: непрерывный штраф за недолёт
        i = int(np.argmax(sol.y[0]))
        F1 = sol.y[1][i] - SLIT_MID - (WALL_X - sol.y[0][i])
    return [F1, x_land - TARGET_X]


def auto_shoot(law, v_T, w0, shear, v_lim, a_lim):
    """Пристрелка: метод Ньютона от школьного прицела,
    при неудаче — грубая сетка и Ньютон от лучшей её точки."""
    log = []

    def acceptable(v0, ang):
        if not (v_lim[0] <= v0 <= v_lim[1] and a_lim[0] <= ang <= a_lim[1]):
            return False
        r = analyse(integrate(v0, ang, law, v_T, w0, shear))
        return r['hit'] and not r['crashed']

    seed = np.array(school_aim())
    log.append("начальное приближение — школьный прицел: "
               "v0 = %.2f м/с, угол = %.2f°" % (seed[0], seed[1]))
    res = root(residual, seed, args=(law, v_T, w0, shear), method='hybr', tol=1e-7)
    if res.success and acceptable(*res.x):
        log.append("метод Ньютона сошёлся за %d вычислений траектории" % res.nfev)
        return res.x[0], res.x[1], log

    log.append("от школьного прицела не сошлось — грубый поиск по сетке")
    best, best_val = None, np.inf
    for v0 in np.linspace(v_lim[0], v_lim[1], 18):
        for ang in np.linspace(a_lim[0], a_lim[1], 18):
            F = residual([v0, ang], law, v_T, w0, shear, rtol=1e-5)
            val = max(abs(F[0]), abs(F[1]))
            if val < best_val:
                best_val, best = val, (v0, ang)
    log.append("лучшая точка сетки: v0 = %.2f, угол = %.2f (невязка %.2f м)"
               % (best[0], best[1], best_val))
    res = root(residual, np.array(best), args=(law, v_T, w0, shear),
               method='hybr', tol=1e-7)
    if res.success and acceptable(*res.x):
        log.append("Ньютон от лучшей точки сетки сошёлся за %d вычислений" % res.nfev)
        return res.x[0], res.x[1], log

    log.append("в заданных диапазонах v0 и угла попадание недостижимо")
    return None, None, log


print("движок загружен: стена x = %.0f, бойница %.0f..%.0f м, цель x = %.0f м"
      % (WALL_X, SLIT_LO, SLIT_HI, TARGET_X))

In [ ]:
# =============================================================
#  БРОСОК КАМНЯ — интерфейс.
#  Требует предыдущую ячейку (движок). Запускать после неё.
# =============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

V_LIM = (8.0, 40.0)          # диапазон начальной скорости, м/с
A_LIM = (10.0, 89.0)         # диапазон угла, градусы
CONTINUOUS = True            # False -- если на слабой машине слайдеры дёргаются
N_SWEEP = 31                 # точек в графике функции промаха

_SL = dict(continuous_update=CONTINUOUS, readout_format='.2f',
           style={'description_width': '132px'},
           layout=widgets.Layout(width='340px'))

_EQ = {
    LAW_NONE: r"$\dot{\vec{v}} = \vec{g}$",
    LAW_LIN:  r"$\dot{\vec{v}} = \vec{g} - \beta\,(\vec{v}-\vec{w})$",
    LAW_QUAD: r"$\dot{\vec{v}} = \vec{g} - \frac{k}{m}\,"
              r"|\vec{v}-\vec{w}|\,(\vec{v}-\vec{w})$",
}
_LAW_TXT = {LAW_NONE: 'нет трения', LAW_LIN: 'линейное', LAW_QUAD: 'квадратичное'}


class StoneThrowApp:

    def __init__(self):
        self._busy = False
        self._sweep_key = None
        self._sweep = None
        self._legend_key = None
        self._log = ''
        self._build_controls()
        self._build_figure()
        self._wire()
        display(self.panel, self.out, self.info)
        self._refresh()

    # ---------------- органы управления ----------------
    def _build_controls(self):
        self.w_law = widgets.ToggleButtons(
            options=[('нет трения', LAW_NONE), ('линейное', LAW_LIN),
                     ('квадратичное', LAW_QUAD)],
            value=LAW_QUAD, description='Трение:',
            style={'description_width': '70px', 'button_width': '128px'})
        self.w_vT = widgets.FloatSlider(value=22.1, min=12.0, max=60.0, step=0.5,
                                        description='v_T (терм.), м/с:', **_SL)
        self.w_v0 = widgets.FloatSlider(value=20.0, min=V_LIM[0], max=V_LIM[1],
                                        step=0.1, description='скорость v0, м/с:', **_SL)
        self.w_ang = widgets.FloatSlider(value=45.0, min=A_LIM[0], max=A_LIM[1],
                                         step=0.25, description='угол, град:', **_SL)
        self.w_w0 = widgets.FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.5,
                                        description='ветер w0, м/с:', **_SL)
        self.w_sh = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05,
                                        description='сдвиг dw/dy, 1/с:', **_SL)
        self.w_par = widgets.Checkbox(value=True, indent=False,
                                      description='парабола без трения')
        self.w_miss = widgets.Checkbox(value=True, indent=False,
                                       description='функция промаха')
        self.b_school = widgets.Button(description='Школьный прицел',
                                       tooltip='Точный ответ задачи при k = 0',
                                       layout=widgets.Layout(width='170px'))
        self.b_shoot = widgets.Button(description='Пристреляться',
                                      button_style='info',
                                      tooltip='Метод стрельбы: решить краевую задачу',
                                      layout=widgets.Layout(width='170px'))
        self.b_reset = widgets.Button(description='Сброс',
                                      layout=widgets.Layout(width='90px'))
        self.out = widgets.Output()
        self.info = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML("<h3 style='margin:2px 0'>Бросок камня: "
                         "пройти в бойницу и попасть в цель</h3>"),
            self.w_law,
            widgets.HBox([self.w_v0, self.w_ang]),
            widgets.HBox([self.w_vT, self.w_w0, self.w_sh]),
            widgets.HBox([self.b_school, self.b_shoot, self.b_reset,
                          self.w_par, self.w_miss]),
        ])

    def _wire(self):
        for w in (self.w_law, self.w_vT, self.w_v0, self.w_ang,
                  self.w_w0, self.w_sh, self.w_par, self.w_miss):
            w.observe(self._on_change, names='value')
        self.b_school.on_click(self._on_school)
        self.b_shoot.on_click(self._on_shoot)
        self.b_reset.on_click(self._on_reset)

    # ---------------- фигура строится ОДИН раз ----------------
    def _build_figure(self):
        self.fig, (self.ax, self.ax2) = plt.subplots(
            1, 2, figsize=(11.2, 4.6), dpi=88,
            gridspec_kw={'width_ratios': [2.85, 1.0]})
        self.fig.subplots_adjust(left=0.055, right=0.985, top=0.91,
                                 bottom=0.115, wspace=0.28)
        plt.close(self.fig)          # чтобы фигура не дублировалась под ячейкой

        ax = self.ax
        ax.axhline(0.0, color='#5a4632', lw=1.2, zorder=1)
        self.a_wall_lo, = ax.plot([WALL_X, WALL_X], [0, SLIT_LO],
                                  color='saddlebrown', lw=7, solid_capstyle='butt',
                                  zorder=3)
        self.a_wall_hi, = ax.plot([WALL_X, WALL_X], [SLIT_HI, 20],
                                  color='saddlebrown', lw=7, solid_capstyle='butt',
                                  zorder=3)
        self.a_slit, = ax.plot([WALL_X, WALL_X], [SLIT_LO, SLIT_HI],
                               color='#7fc7e8', lw=7, solid_capstyle='butt',
                               label='бойница', zorder=3)
        self.a_target, = ax.plot([TARGET_X - TARGET_R, TARGET_X + TARGET_R], [0, 0],
                                 color='crimson', lw=9, solid_capstyle='round',
                                 label='цель', zorder=4)
        self.a_traj, = ax.plot([], [], color='#1f4e9c', lw=2.6,
                               label='траектория (численно)', zorder=6)
        self.a_par, = ax.plot([], [], color='#888888', lw=1.6, ls='--',
                              label='парабола (k = 0)', zorder=5)
        # точное решение показываем кружками ПОВЕРХ численной кривой:
        # если они легли на линию — решатель проверен
        self.a_exact, = ax.plot([], [], ls='none', marker='o', ms=5.0,
                                mfc='none', mec='#ff6a00', mew=1.5,
                                label='точное решение (проверка)', zorder=7)
        # маркеры удара и попадания в легенду не идут: они говорят сами за себя,
        # а их появление/исчезновение заставляло пересобирать легенду каждый кадр
        self.a_crash, = ax.plot([], [], 'X', color='darkred', ms=13,
                                label='_nolegend_', zorder=8)
        self.a_hit, = ax.plot([], [], '*', color='green', ms=18,
                              label='_nolegend_', zorder=8)
        self.t_eq = ax.text(0.015, 0.955, '', transform=ax.transAxes,
                            fontsize=13, va='top', ha='left',
                            bbox=dict(fc='white', ec='#cccccc', alpha=0.9,
                                      boxstyle='round,pad=0.35'))
        ax.set_xlabel('x, м')
        ax.set_ylabel('y, м')
        ax.grid(True, ls='--', alpha=0.45)
        ax.set_adjustable('box')

        a2 = self.ax2
        a2.axhline(0.0, color='black', lw=1.0)
        self.a_m_slit, = a2.plot([], [], color='#1f7fb0', lw=2.0,
                                 label='промах по бойнице')
        self.a_m_targ, = a2.plot([], [], color='crimson', lw=2.0,
                                 label='промах по цели')
        self.a_m_now = a2.axvline(45.0, color='#444444', lw=1.2, ls='--')
        a2.set_xlabel('угол, град')
        a2.set_ylabel('промах, м')
        a2.set_xlim(*A_LIM)
        a2.grid(True, ls='--', alpha=0.45)
        a2.set_title('промах: нужен ноль у обеих', fontsize=9.5)
        a2.legend(fontsize=7.5, loc='upper right')

    # ---------------- функция промаха (с кэшем) ----------------
    def _sweep_curves(self, law, v_T, v0, w0, sh):
        key = (law, v_T, v0, w0, sh)
        if key == self._sweep_key:
            return self._sweep
        angs = np.linspace(A_LIM[0], A_LIM[1], N_SWEEP)
        f_slit, f_targ = [], []
        for a in angs:
            sol = integrate(v0, a, law, v_T, w0, sh, rtol=1e-5)
            f_slit.append(sol.y_events[1][0][1] - SLIT_MID
                          if len(sol.y_events[1]) else np.nan)
            f_targ.append(sol.y_events[0][0][0] - TARGET_X
                          if (sol.status == 1 and len(sol.y_events[0])) else np.nan)
        self._sweep_key = key
        self._sweep = (angs, np.array(f_slit), np.array(f_targ))
        return self._sweep

    # ---------------- отрисовка ----------------
    def render(self):
        law = self.w_law.value
        v_T, v0, ang = self.w_vT.value, self.w_v0.value, self.w_ang.value
        w0, sh = self.w_w0.value, self.w_sh.value
        self.w_vT.disabled = (law == LAW_NONE)

        r = analyse(integrate(v0, ang, law, v_T, w0, sh))

        # --- траектория ---
        self.a_traj.set_data(r['x'], r['y'])
        handles = [self.a_traj]

        par_xy = None
        if self.w_par.value and law != LAW_NONE:
            par_xy = parabola(v0, ang)
            self.a_par.set_data(*par_xy)
            handles.append(self.a_par)
        else:
            self.a_par.set_data([], [])

        # точное решение: k=0 -- парабола; линейное без сдвига -- замкнутая формула
        if law == LAW_LIN and sh == 0.0:
            xe, ye = linear_exact(v0, ang, v_T, w0, r['t_end'], n=17)
            self.a_exact.set_data(xe, ye)
            handles.append(self.a_exact)
        else:
            self.a_exact.set_data([], [])

        self.a_crash.set_data([WALL_X], [r['y_crash']]) if r['crashed']             else self.a_crash.set_data([], [])
        self.a_hit.set_data([r['x_land']], [0.0]) if r['hit']             else self.a_hit.set_data([], [])
        handles += [self.a_slit, self.a_target]

        # --- кадр подстраивается под траекторию ---
        x_hi = max(30.0, float(np.max(r['x'])) + 3.0)
        x_lo = min(-1.0, float(np.min(r['x'])) - 2.0)
        y_hi = max(14.0, r['apex'] + 3.0)
        if par_xy is not None:      # парабола тоже должна влезать, но не любой
            x_cap = max(32.0, float(np.max(r['x'])) * 1.5)   # ценой: сцена важнее
            y_cap = max(15.0, r['apex'] * 1.7)
            x_hi = min(max(x_hi, float(par_xy[0].max()) + 2.0), x_cap)
            y_hi = min(max(y_hi, float(par_xy[1].max()) + 2.0), y_cap)
        self.ax.set_xlim(x_lo, x_hi)
        self.ax.set_ylim(-1.0, y_hi)
        self.a_wall_hi.set_data([WALL_X, WALL_X], [SLIT_HI, y_hi])

        self.t_eq.set_text(_EQ[law] + ('' if law == LAW_NONE or (w0 == 0 and sh == 0)
                                       else '\n' + r'$\vec{w}=(w_0+\gamma y,\;0)$'))

        if r['crashed']:
            ttl, col = ('Камень разбился о стену: пришёл на %.2f м, '
                        'нужно %.0f–%.0f м' % (r['y_crash'], SLIT_LO, SLIT_HI),
                        'darkred')
        elif r['hit']:
            ttl, col = ('✓ Точное попадание: приземление на %.2f м'
                        % r['x_land'], 'green')
        elif r['x_land'] is None:
            ttl, col = 'Камень не приземлился за отведённое время', '#333333'
        else:
            ttl, col = ('Мимо цели на %.2f м (приземление %.2f м)'
                        % (abs(r['x_land'] - TARGET_X), r['x_land']), '#333333')
        self.ax.set_title(ttl, fontsize=12.5, color=col, pad=8)
        key = tuple(id(h) for h in handles)      # легенду пересобираем
        if key != self._legend_key:               # только когда состав сменился
            self.ax.legend(handles=handles, loc='upper right', fontsize=8.5)
            self._legend_key = key

        # --- функция промаха ---
        if self.w_miss.value:
            self.ax2.set_visible(True)
            angs, f_s, f_t = self._sweep_curves(law, v_T, v0, w0, sh)
            self.a_m_slit.set_data(angs, f_s)
            self.a_m_targ.set_data(angs, f_t)
            self.a_m_now.set_xdata([ang, ang])
            fin = np.concatenate([f_s[np.isfinite(f_s)], f_t[np.isfinite(f_t)]])
            m = 12.0 if fin.size == 0 else min(30.0, max(4.0, np.abs(fin).max() * 1.1))
            self.ax2.set_ylim(-m, m)
        else:
            self.ax2.set_visible(False)

        return self._info_html(law, v_T, v0, ang, w0, sh, r)

    # ---------------- числа под графиком ----------------
    def _info_html(self, law, v_T, v0, ang, w0, sh, r):
        def cell(ok, txt):
            return ("<b style='color:%s'>%s</b>"
                    % ('green' if ok else '#b00000', txt))

        if law == LAW_NONE:
            par = 'k = 0'
            solvable = ('<b style='
                        '"color:green">да</b> — траектория есть парабола, '
                        'задача решается школьной формулой')
        elif law == LAW_LIN:
            par = ('&beta; = g/v_T = %.4f 1/с,&nbsp; v_T = %.1f м/с'
                   % (G / v_T, v_T))
            solvable = ('<b style="color:green">да</b> — система распадается на '
                        'два <i>независимых</i> линейных уравнения: '
                        'v<sub>x</sub>&prime; = &minus;&beta;(v<sub>x</sub>&minus;w), '
                        'v<sub>y</sub>&prime; = &minus;g &minus; &beta;v<sub>y</sub>'
                        if sh == 0.0 else
                        '<b style="color:#b06000">почти</b> — уравнения линейные, но '
                        'сдвиг ветра связывает x с y: в правой части по x стоит y')
        else:
            par = ('k/m = g/v_T&sup2; = %.4f 1/м,&nbsp; v_T = %.1f м/с'
                   % (G / v_T ** 2, v_T))
            solvable = ('<b style="color:#b00000">нет</b> — |<b>v</b>| смешивает '
                        'компоненты, замкнутой формулы не существует. '
                        'Остаётся численное решение и качественный анализ')

        y_w = ('не достал до стены' if r['y_wall'] is None else
               cell(SLIT_LO <= r['y_wall'] <= SLIT_HI, '%.2f м' % r['y_wall']))
        x_l = ('—' if r['x_land'] is None else
               cell(abs(r['x_land'] - TARGET_X) <= TARGET_R, '%.2f м' % r['x_land']))
        note = ''
        if law == LAW_NONE and (w0 != 0.0 or sh != 0.0):
            note = ("<div style='color:#b06000;margin-top:4px'>При k = 0 ветер "
                    "в уравнение не входит: он действует только через трение. "
                    "Ползунки ветра сейчас ни на что не влияют — это не баг.</div>")
        log = ("<div style='color:#555;margin-top:4px;font-family:monospace;"
               "font-size:11px'>%s</div>" % self._log) if self._log else ''

        return (
            "<div style='font-size:13px;line-height:1.65;max-width:1120px'>"
            "<table style='border-collapse:collapse'><tr>"
            "<td style='padding-right:22px'>высота на стене: %s"
            " <span style='color:#777'>(нужно %.0f–%.0f)</span></td>"
            "<td style='padding-right:22px'>приземление: %s"
            " <span style='color:#777'>(цель %.0f–%.0f)</span></td>"
            "<td style='padding-right:22px'>апогей: %.2f м</td>"
            "<td style='padding-right:22px'>время полёта: %.2f с</td>"
            "<td>скорость в конце: %.1f м/с</td>"
            "</tr></table>"
            "<div>%s (%s)</div>"
            "<div>решается в замкнутом виде? %s</div>%s%s</div>"
            % (y_w, SLIT_LO, SLIT_HI, x_l, TARGET_X - TARGET_R, TARGET_X + TARGET_R,
               r['apex'], r['t_end'], r['v_end'],
               _LAW_TXT[law], par, solvable, note, log))

    # ---------------- реакция на события ----------------
    def _refresh(self):
        html = self.render()
        self.info.value = html
        with self.out:
            clear_output(wait=True)
            display(self.fig)

    def _on_change(self, change=None):
        if not self._busy:
            self._log = ''
            self._refresh()

    def _set(self, **kw):
        self._busy = True
        try:
            for name, val in kw.items():
                getattr(self, name).value = val
        finally:
            self._busy = False
        self._refresh()

    def _on_school(self, _b=None):
        v0, ang = school_aim()
        self._log = ('школьный прицел (замкнутая формула при k = 0): '
                     'v0 = %.2f м/с, угол = %.2f°' % (v0, ang))
        self._set(w_v0=round(v0, 2), w_ang=round(ang, 2))

    def _on_shoot(self, _b=None):
        self.b_shoot.description = 'считаю...'
        self.b_shoot.disabled = True
        try:
            v0, ang, log = auto_shoot(self.w_law.value, self.w_vT.value,
                                      self.w_w0.value, self.w_sh.value,
                                      V_LIM, A_LIM)
            self._log = ' &rarr; '.join(log)
            if v0 is None:
                self._refresh()
            else:
                self._set(w_v0=round(v0, 2), w_ang=round(ang, 2))
        finally:
            self.b_shoot.description = 'Пристреляться'
            self.b_shoot.disabled = False

    def _on_reset(self, _b=None):
        self._log = ''
        self._set(w_law=LAW_QUAD, w_vT=22.1, w_v0=20.0, w_ang=45.0,
                  w_w0=0.0, w_sh=0.0)


app = StoneThrowApp()

---

## Вопросы к виджету

1. **Поставьте «нет трения» и покрутите ползунки ветра.** Траектория не меняется. Почему это правильно, а не ошибка в программе?

2. **При линейном трении** оранжевые кружки (замкнутая формула) ложатся на синюю кривую (численное решение). Почему для квадратичного трения такую формулу выписать нельзя? Ответ — в одном множителе.

3. **Пристреляйтесь при линейном трении, потом при квадратичном, не меняя $v_T$.** Прицелы получаются разные: 22,75 м/с и $46{,}8^\circ$ против 20,22 м/с и $51{,}7^\circ$. Значит, разница между законами — не в «силе трения», а в самом законе.

4. **$v_T$ — терминальная скорость свободного падения**, и коэффициенты считаются из неё: $\beta = g/v_T$ для линейного закона, $k/m = g/v_T^2$ для квадратичного. Проверьте это, приравняв нулю правую часть одномерного уравнения падения.

5. **Правая панель.** Почему два условия нельзя выполнить, крутя один угол? Сколько параметров нужно и почему именно столько?

6. **Сдвиг ветра $dw/dy \ne 0$** делает горизонтальную силу зависящей от высоты. Что при этом происходит с «независимостью» уравнений для $x$ и $y$ из средней строки таблицы?

## Жёлтая рамка (необязательное, на дом)

Оцените $k/m$ для настоящего камня: масса 200 г, радиус 3 см, $C_d \approx 0{,}47$, $\rho = 1{,}2$ кг/м³, и найдите его терминальную скорость. Поставьте её ползунком и проверьте школьный прицел ещё раз.

<details>
<summary>Ответ</summary>

$k/m = \dfrac{\rho C_d \pi r^2}{2m} \approx 0{,}004$ 1/м, отсюда $v_T = \sqrt{g/(k/m)} \approx 50$ м/с. При таком $v_T$ школьный прицел **проходит** в бойницу (на высоте 8,6 м), но до цели не долетает 2 м.

Значение по умолчанию $v_T = 22$ м/с отвечает $k/m \approx 0{,}02$ 1/м — это в 5 раз больше, чем у камня, скорее лёгкий мяч. Взято, чтобы эффект был виден на глаз. Вывод «для настоящего камня на дистанции 25 м трение почти незаметно» — тоже содержательный: он объясняет, почему школьная формула вообще работает.
</details>

---

*Что можно править в коде:* геометрия сцены (`WALL_X`, `SLIT_LO`, `SLIT_HI`, `TARGET_X`, `TARGET_R`) — в начале ячейки с движком; диапазоны ползунков (`V_LIM`, `A_LIM`) и переключатель `CONTINUOUS` (плавное обновление при протяжке ползунка; поставьте `False`, если на слабой машине картинка дёргается) — в начале ячейки с интерфейсом.